# Resolución de Sudokus — Modelo de capas densas (MLP)

Entrena una red **totalmente conectada** que aprende a resolver sudokus a partir
del dataset `data/sudoku.csv` (9.000.000 de pares `puzzle,solution`).

- **Entrada:** tablero 9×9 aplanado → 81 valores (cada celda 0–9, normalizada `/9`).
- **Salida:** 81×9 = 729 → por cada casilla, la probabilidad de cada dígito 1–9.
- **Oráculo / referencia:** el backtracking de `src/solver.py` (siempre correcto).

> Nota honesta: un MLP con solo capas densas **no captura bien** las restricciones
> fila/columna/bloque del sudoku. Sirve para aprender el montaje, pero su acierto
> por *tablero completo* será bajo comparado con el backtracking.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

ROOT = Path.cwd().parent
CSV = ROOT / "data" / "sudoku.csv"
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device, "| csv:", CSV.exists())

device: mps | csv: True


## 1. Muestra aleatoria de 1.000.000

El CSV tiene 9M de filas. Elegimos al azar qué filas conservar y leemos el fichero
por trozos (chunks), para no cargar 1,4 GB en memoria de golpe.

In [2]:
N_TOTAL  = 9_000_000
N_SAMPLE = 1_000_000
rng = np.random.default_rng(42)

keep = np.zeros(N_TOTAL, dtype=bool)
keep[rng.choice(N_TOTAL, N_SAMPLE, replace=False)] = True   # 1M filas marcadas

puzzles, solutions = [], []
start = 0
t0 = time.time()
for chunk in pd.read_csv(CSV, chunksize=1_000_000, dtype=str):
    n = len(chunk)
    sub = chunk[keep[start:start + n]]
    puzzles.append(sub["puzzle"].to_numpy())
    solutions.append(sub["solution"].to_numpy())
    start += n
puzzles   = np.concatenate(puzzles)
solutions = np.concatenate(solutions)
print(f"muestreados {len(puzzles):,} sudokus en {time.time()-t0:.1f}s")

muestreados 1,000,000 sudokus en 8.4s


## 2. Codificar a tensores

Convertimos cada string de 81 caracteres en un vector de 81 enteros (truco rápido
con `np.frombuffer`). La entrada se normaliza `/9`; la salida son clases 0–8
(dígito − 1) para usar `CrossEntropyLoss`.

In [3]:
def a_rejilla(strs):
    """(N,) strings de 81 chars -> (N,81) enteros."""
    a = np.frombuffer("".join(strs.tolist()).encode("ascii"), dtype=np.uint8) - ord("0")
    return a.reshape(-1, 81)

P = a_rejilla(puzzles)      # (N,81) valores 0-9 (0 = vacia)
S = a_rejilla(solutions)    # (N,81) valores 1-9

X = torch.from_numpy((P / 9.0).astype(np.float32))   # (N,81) en [0,1]
Y = torch.from_numpy((S - 1).astype(np.int64))       # (N,81) clases 0-8

# barajar y separar validacion
perm = torch.randperm(len(X))
X, Y = X[perm], Y[perm]
n_val = 50_000
Xtr, Ytr = X[:-n_val], Y[:-n_val]
Xva, Yva = X[-n_val:], Y[-n_val:]
print("train:", Xtr.shape, "| val:", Xva.shape)

train_dl = DataLoader(TensorDataset(Xtr, Ytr), batch_size=512, shuffle=True)

train: torch.Size([950000, 81]) | val: torch.Size([50000, 81])


## 3. Modelo (solo capas densas)

`nn.Linear` es la capa densa. Encadenamos: entrada **81** → neuronas **512, 1024,
512** → salida **729**, que se reorganiza en **(81, 9)**.

In [4]:
class DenseSudokuNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(81, 512),   nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 1024), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(1024, 512), nn.ReLU(),
            nn.Linear(512, 729),
        )

    def forward(self, x):                 # x: (N, 81)
        return self.net(x).view(-1, 81, 9)   # (N, 81, 9)


model = DenseSudokuNet().to(device)
print(model)

DenseSudokuNet(
  (net): Sequential(
    (0): Linear(in_features=81, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=1024, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=1024, out_features=512, bias=True)
    (7): ReLU()
    (8): Linear(in_features=512, out_features=729, bias=True)
  )
)


## 4. Entrenamiento

`CrossEntropyLoss` sobre las 81 casillas a la vez: aplanamos la salida a `(N·81, 9)`
y el objetivo a `(N·81,)`.

In [5]:
EPOCHS = 8
opt  = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluar(Xv, Yv, bs=8192):
    model.eval()
    cell_ok = board_ok = 0
    for i in range(0, len(Xv), bs):
        pred = model(Xv[i:i+bs].to(device)).argmax(-1).cpu()   # (b,81)
        eq = pred == Yv[i:i+bs]
        cell_ok  += eq.sum().item()
        board_ok += eq.all(1).sum().item()
    return cell_ok / (len(Xv) * 81), board_ok / len(Xv)

for ep in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        loss = crit(model(xb).reshape(-1, 9), yb.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    acc_cel, acc_tab = evaluar(Xva, Yva)
    print(f"epoch {ep}/{EPOCHS}  loss {loss.item():.3f}  "
          f"acc_celda {acc_cel:.3f}  acc_tablero {acc_tab:.3f}  ({time.time()-t0:.0f}s)")

epoch 1/8  loss 1.996  acc_celda 0.230  acc_tablero 0.000  (11s)


epoch 2/8  loss 1.957  acc_celda 0.268  acc_tablero 0.000  (7s)


epoch 3/8  loss 1.926  acc_celda 0.300  acc_tablero 0.000  (7s)


epoch 4/8  loss 1.903  acc_celda 0.317  acc_tablero 0.000  (7s)


epoch 5/8  loss 1.866  acc_celda 0.328  acc_tablero 0.000  (8s)


epoch 6/8  loss 1.832  acc_celda 0.331  acc_tablero 0.000  (9s)


epoch 7/8  loss 1.793  acc_celda 0.325  acc_tablero 0.000  (8s)


epoch 8/8  loss 1.753  acc_celda 0.324  acc_tablero 0.000  (8s)


## 5. Guardar el modelo

In [6]:
dst = ROOT / "models" / "sudoku_solver_mlp.pt"
torch.save(model.state_dict(), dst)
print("guardado en", dst)

guardado en /Users/nora/Documentos_Clase/BOOTCAMP/sudoku/models/sudoku_solver_mlp.pt


## 6. Comparar con el solver (oráculo)

Resolvemos un puzzle con la red y con el backtracking de `src/solver.py`, y medimos
con `evaluar_solucion`. Así se ve la diferencia entre el modelo aprendido y la
solución algorítmica exacta.

In [7]:
import sys
sys.path.append(str(ROOT / "src"))
import solver

@torch.no_grad()
def resolver_con_red(puzzle81):
    x = torch.tensor(puzzle81, dtype=torch.float32).view(1, 81) / 9.0
    pred = model(x.to(device)).argmax(-1).cpu().numpy().reshape(9, 9) + 1
    return pred.tolist()

ej = P[0]                                   # un puzzle de la muestra
tablero = ej.reshape(9, 9).tolist()
red = resolver_con_red(ej)
bt  = solver.solve(tablero)

print("Puzzle:");          solver.print_board(tablero)
print("\nRed (densa):");  solver.print_board(red)
print("\nSolver (backtracking):"); solver.print_board(bt)
print("\nMetricas red vs solver:", solver.evaluar_solucion(red, bt))

Puzzle:
. 7 . | . . . | . 4 3
. 4 . | . . 9 | 6 1 .
8 . . | 6 3 4 | 9 . .
------+-------+------
. 9 4 | . 5 2 | . . .
3 5 8 | 4 6 . | . 2 .
. . . | 8 . . | 5 3 .
------+-------+------
. 8 . | . 7 . | . 9 1
9 . 2 | 1 . . | . . 5
. . 7 | . 4 . | 8 . 2

Red (densa):
9 7 9 | 9 9 9 | 3 4 4
9 4 9 | 1 9 8 | 6 5 5
7 2 1 | 5 3 5 | 9 1 2
------+-------+------
4 9 4 | 2 5 3 | 4 1 7
2 5 7 | 4 7 2 | 1 9 9
1 1 1 | 8 1 4 | 5 5 9
------+-------+------
4 7 2 | 2 7 5 | 1 9 1
8 1 2 | 4 9 9 | 2 3 5
3 1 7 | 3 9 9 | 7 5 4

Solver (backtracking):
6 7 9 | 5 1 8 | 2 4 3
5 4 3 | 7 2 9 | 6 1 8
8 2 1 | 6 3 4 | 9 5 7
------+-------+------
7 9 4 | 3 5 2 | 1 8 6
3 5 8 | 4 6 1 | 7 2 9
2 1 6 | 8 9 7 | 5 3 4
------+-------+------
4 8 5 | 2 7 6 | 3 9 1
9 6 2 | 1 8 3 | 4 7 5
1 3 7 | 9 4 5 | 8 6 2

Metricas red vs solver: {'celdas_correctas': 26, 'total': 81, 'precision': 0.32098765432098764, 'exacta': False, 'solucion_valida': False}
